[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maercaestro/megat/blob/main/notebooks/04_inference_testing.ipynb)

# Notebook 04 — Megat GPT-2 355M Inference & Testing

Load the pretrained Megat model from HuggingFace Hub and generate Bahasa Melayu text.

**Model:** `maercaestro/megat-gpt2-355m`  
**Architecture:** GPT-2 Medium (355M), trained from scratch on FineWeb2 Malay  
**Tokenizer:** Custom 32k BPE tokenizer trained on Malay corpus

---
**This notebook:**
1. Installs dependencies
2. Downloads model + tokenizer from HF Hub
3. Loads the model into memory
4. Runs generation with a range of prompts and settings

## Step 1 — Install dependencies

In [ ]:
!pip install -q tokenizers huggingface_hub

## Step 2 — Clone the repo (for model.py) and download model weights

In [ ]:
import os

# Clone the training repo so we can import model.py
if not os.path.exists('megat'):
    !git clone https://github.com/maercaestro/megat.git
else:
    print('Repo already cloned.')

# Add repo to path so we can import model.py
import sys
sys.path.insert(0, 'megat')

In [ ]:
from huggingface_hub import snapshot_download

MODEL_REPO = 'maercaestro/megat-gpt2-355m'
MODEL_DIR  = 'megat_model'

print(f'Downloading model from {MODEL_REPO} ...')
snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=MODEL_DIR,
)
print('Download complete.')
print('Files:', os.listdir(MODEL_DIR))

## Step 3 — Load the tokenizer

In [ ]:
from tokenizers import ByteLevelBPETokenizer

# Locate tokenizer files — check common paths inside the downloaded dir
def find_tokenizer(base_dir):
    candidates = [
        base_dir,
        os.path.join(base_dir, 'megat_tokenizer'),
        os.path.join(base_dir, 'tokenizer'),
    ]
    for d in candidates:
        vocab  = os.path.join(d, 'vocab.json')
        merges = os.path.join(d, 'merges.txt')
        if os.path.exists(vocab) and os.path.exists(merges):
            return vocab, merges
    raise FileNotFoundError(
        f'Could not find vocab.json + merges.txt under {base_dir}.\n'
        f'Contents: {os.listdir(base_dir)}'
    )

vocab_path, merges_path = find_tokenizer(MODEL_DIR)
print(f'Tokenizer found:\n  vocab:  {vocab_path}\n  merges: {merges_path}')

tokenizer = ByteLevelBPETokenizer(vocab_path, merges_path)

EOT_ID   = tokenizer.token_to_id('<|endoftext|>')
STORY_ID = tokenizer.token_to_id('<|story|>')

print(f'Vocab size:         {tokenizer.get_vocab_size():,}')
print(f'<|endoftext|> ID:   {EOT_ID}')
print(f'<|story|> ID:       {STORY_ID}')

# Quick sanity check — common Malay words should be 1–2 tokens
test_words = ['masyarakat', 'pembangunan', 'kerajaan', 'negara', 'Malaysia']
print('\nTokenization check:')
for w in test_words:
    ids = tokenizer.encode(w).ids
    print(f'  {w:20s} → {len(ids)} token(s): {ids}')

## Step 4 — Load the model

In [ ]:
import torch
from model import GPT, GPTConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Locate the checkpoint file
def find_checkpoint(base_dir):
    for fname in os.listdir(base_dir):
        if fname.endswith('.pt') or fname.endswith('.pth') or fname.endswith('.bin'):
            # skip tokenizer bin files
            if 'token' not in fname and 'vocab' not in fname:
                return os.path.join(base_dir, fname)
    # Also check subdirs
    for root, dirs, files in os.walk(base_dir):
        for fname in files:
            if fname.endswith('.pt') or fname.endswith('.pth'):
                return os.path.join(root, fname)
    raise FileNotFoundError(
        f'No checkpoint (.pt/.pth) found in {base_dir}.\n'
        f'Contents: {os.listdir(base_dir)}'
    )

ckpt_path = find_checkpoint(MODEL_DIR)
print(f'Loading checkpoint: {ckpt_path}')

checkpoint = torch.load(ckpt_path, map_location=device)
print(f'Checkpoint keys: {list(checkpoint.keys()) if isinstance(checkpoint, dict) else type(checkpoint)}')

In [ ]:
# Build GPTConfig — must match training config exactly
config = GPTConfig(
    block_size = 1024,
    vocab_size  = tokenizer.get_vocab_size(),
    n_layer     = 24,
    n_head      = 16,
    n_embd      = 1024,
)

model = GPT(config)

# Load state dict — handle both raw state_dict and wrapped checkpoint formats
if isinstance(checkpoint, dict) and 'model' in checkpoint:
    state_dict = checkpoint['model']
    step       = checkpoint.get('step', '?')
    val_loss   = checkpoint.get('val_loss', '?')
    print(f'Checkpoint from step {step}, val_loss={val_loss}')
elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint  # raw state dict

# Strip 'module.' prefix if the model was saved with DataParallel
state_dict = {k.replace('_orig_mod.', '').replace('module.', ''): v
              for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nModel loaded successfully!')
print(f'Parameters: {total_params/1e6:.1f}M')
print(f'Device:     {device}')

## Step 5 — Generation helper

In [ ]:
def generate(prompt, max_new_tokens=200, temperature=0.8, top_p=0.95, num_samples=1):
    """
    Generate text given a prompt string.
    
    Args:
        prompt:         Input text to condition on
        max_new_tokens: Number of tokens to generate
        temperature:    Sampling temperature (0.7–0.9 for fiction)
        top_p:          Nucleus sampling threshold
        num_samples:    How many independent completions to generate
    """
    token_ids = tokenizer.encode(prompt).ids
    idx = torch.tensor([token_ids] * num_samples, dtype=torch.long, device=device)

    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens,
                             temperature=temperature, top_p=top_p)

    results = []
    for i in range(num_samples):
        generated_ids = out[i, len(token_ids):].tolist()
        # Stop at EOT token if encountered
        if EOT_ID in generated_ids:
            generated_ids = generated_ids[:generated_ids.index(EOT_ID)]
        generated_text = tokenizer.decode(generated_ids)
        results.append(prompt + generated_text)

    return results


def print_generation(prompt, **kwargs):
    num_samples = kwargs.get('num_samples', 1)
    print('─' * 60)
    print(f'PROMPT: {prompt!r}')
    print('─' * 60)
    results = generate(prompt, **kwargs)
    for i, text in enumerate(results):
        if num_samples > 1:
            print(f'[Sample {i+1}]')
        print(text)
        if num_samples > 1:
            print()
    print()

print('Generation helper ready.')

## Step 6 — Test generations

### 6a. Open-ended continuations

In [ ]:
print_generation(
    'Pada suatu petang yang tenang,',
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.95,
)

In [ ]:
print_generation(
    'Dia tidak pernah menyangka bahawa hari itu akan mengubah hidupnya selamanya.',
    max_new_tokens=250,
    temperature=0.85,
    top_p=0.95,
)

In [ ]:
print_generation(
    'Hujan turun dengan lebat ketika',
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.92,
)

### 6b. Multiple samples from the same prompt

Useful for seeing the range of what the model can produce.

In [ ]:
print_generation(
    'Di sebuah kampung kecil di tepi hutan,',
    max_new_tokens=150,
    temperature=0.85,
    top_p=0.95,
    num_samples=3,
)

### 6c. Temperature sweep

Low temperature → more predictable and grammatically correct.  
High temperature → more creative but can become incoherent.

In [ ]:
prompt = 'Negara kita sedang menghadapi cabaran'

for temp in [0.5, 0.75, 1.0, 1.2]:
    print(f'\n=== temperature={temp} ===')
    results = generate(prompt, max_new_tokens=100, temperature=temp, top_p=0.95)
    print(results[0])

### 6d. Story format with special tokens

If the model was trained with `<|story|>` delimiters, try prompting with them.

In [ ]:
story_prompt = '<|story|>\nTajuk: Rahsia Di Balik Pintu\nGenre: Cerpen\n\n'

print_generation(
    story_prompt,
    max_new_tokens=300,
    temperature=0.8,
    top_p=0.95,
)

### 6e. Dialogue generation

In [ ]:
print_generation(
    '"Kenapa kamu berbohong kepada aku?" tanya Amir dengan suara yang terketar-ketar.',
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.95,
)

### 6f. Long-form generation

Generate a longer passage by chunking — feed the last 768 tokens as context for each new chunk.

In [ ]:
def generate_long(prompt, total_new_tokens=600, chunk_size=200,
                  context_window=768, temperature=0.8, top_p=0.95):
    """
    Generate longer text by chunking, keeping the last `context_window`
    tokens as context for each new chunk.
    """
    token_ids = tokenizer.encode(prompt).ids
    tokens_generated = 0

    while tokens_generated < total_new_tokens:
        # Use last context_window tokens as input
        context = token_ids[-context_window:]
        idx = torch.tensor([context], dtype=torch.long, device=device)

        this_chunk = min(chunk_size, total_new_tokens - tokens_generated)
        with torch.no_grad():
            out = model.generate(idx, max_new_tokens=this_chunk,
                                 temperature=temperature, top_p=top_p)

        new_ids = out[0, len(context):].tolist()
        # Stop if EOT encountered
        if EOT_ID in new_ids:
            new_ids = new_ids[:new_ids.index(EOT_ID)]
            token_ids.extend(new_ids)
            break

        token_ids.extend(new_ids)
        tokens_generated += len(new_ids)

    return tokenizer.decode(token_ids)


long_prompt = 'Kisah ini bermula pada tahun 1960, ketika Malaysia masih muda sebagai sebuah negara yang baru merdeka.'
print('Generating long-form text (~600 tokens)...')
long_text = generate_long(long_prompt, total_new_tokens=600)
print('─' * 60)
print(long_text)

## Step 7 — Interactive generation

Enter your own prompt and tweak settings.

In [ ]:
# ── Customise these ─────────────────────────────────────
MY_PROMPT       = 'Langit malam itu penuh dengan bintang ketika'
MAX_NEW_TOKENS  = 300
TEMPERATURE     = 0.80
TOP_P           = 0.95
NUM_SAMPLES     = 1
# ────────────────────────────────────────────────────────

print_generation(
    MY_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    num_samples=NUM_SAMPLES,
)

## Step 8 — Model info summary

In [ ]:
print('═' * 50)
print('  Megat GPT-2 355M — Model Summary')
print('═' * 50)
print(f'  Layers:         {config.n_layer}')
print(f'  Attention heads:{config.n_head}')
print(f'  Embedding dim:  {config.n_embd}')
print(f'  Context window: {config.block_size} tokens')
print(f'  Vocab size:     {config.vocab_size:,}')
print(f'  Total params:   {total_params/1e6:.1f}M')
print(f'  Device:         {device}')
print(f'  Checkpoint:     {ckpt_path}')
print('═' * 50)